# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene1407/ML-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: 71% holdout accuracy

The research paper reports approximately 71% holdout accuracy for its machine-learning analysis of growing versus declining content.

My methodology question: How exactly was the growing-versus-declining label constructed? Was it derived from a future performance window, and were all model features available before that window? I would also like to understand whether the holdout split separated related clients or content items. This would help establish how well the reported result represents performance on genuinely unseen data.

Finding 2: Random Forest feature importance

The paper uses Random Forest feature importance to examine which signals are associated with its prediction target.

My methodology question: Were the most important features checked for overlap with the label definition or for shared underlying measurements? Could correlated or label-derived features influence the importance ranking? A feature's importance is useful for describing predictive association, but it does not establish causation.

In [3]:
# Section 1: Load and verify the Week 3 feature frame

import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Load the saved Week 3 feature frame
data_path = Path(
    r"C:\Users\LENOVO\Downloads\ML-1\work\outputs\w03_feature_frame.parquet"
)

feature_df = pd.read_parquet(data_path)

features = [
    "impressions",
    "clicks",
    "avg_position",
    "march_ctr",
    "active_days"
]

target = "position_worsened"

X = feature_df[features].copy()
y = feature_df[target].astype(int).copy()

groups = feature_df["client_hash_id"].copy()

print("Feature frame shape:", feature_df.shape)
print("Model feature shape:", X.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget base rate:", round(y.mean(), 4))

print("\nMissing values:")
print(feature_df[features + [target]].isnull().sum())

print("\nDuplicate rows:", feature_df.duplicated().sum())

print("\nExcluded identifier columns:")
print(["client_hash_id", "content_hash_id"])

Feature frame shape: (158549, 8)
Model feature shape: (158549, 5)

Target distribution:
position_worsened
1    99146
0    59403
Name: count, dtype: int64

Target base rate: 0.6253

Missing values:
impressions          0
clicks               0
avg_position         0
march_ctr            0
active_days          0
position_worsened    0
dtype: int64

Duplicate rows: 0

Excluded identifier columns:
['client_hash_id', 'content_hash_id']


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Validation design

I compare two validation approaches using the same feature set and model configurations.

1. Random row split: randomly divides client-content pairs into training and testing sets. Related records from the same client may appear in both sets.
2. Grouped split: separates records by client using GroupShuffleSplit. A client cannot appear in both training and testing sets.

The random split represents a less restrictive evaluation, while the grouped split tests whether the model generalizes to clients it has not seen during training.

I use Logistic Regression and Random Forest under both designs. I compare accuracy, balanced accuracy, ROC-AUC, average precision, and Precision@50 and Precision@100.

The grouped evaluation is the more relevant test when considering performance on unseen clients. A difference between the two results would indicate sensitivity to the validation design.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Create random and grouped splits

RANDOM_STATE = 42

# Random row split
random_train_idx, random_test_idx = train_test_split(
    np.arange(len(feature_df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Grouped split by client
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

group_train_idx, group_test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

# Verify client overlap
random_train_clients = set(groups.iloc[random_train_idx])
random_test_clients = set(groups.iloc[random_test_idx])

group_train_clients = set(groups.iloc[group_train_idx])
group_test_clients = set(groups.iloc[group_test_idx])

print("Random split:")
print("Training rows:", len(random_train_idx))
print("Testing rows:", len(random_test_idx))
print("Client overlap:", len(random_train_clients & random_test_clients))

print("\nGrouped split:")
print("Training rows:", len(group_train_idx))
print("Testing rows:", len(group_test_idx))
print("Training clients:", len(group_train_clients))
print("Testing clients:", len(group_test_clients))
print("Client overlap:", len(group_train_clients & group_test_clients))

print("\nPositive rate:")
print("Full dataset:", round(y.mean(), 4))
print("Random test:", round(y.iloc[random_test_idx].mean(), 4))
print("Grouped test:", round(y.iloc[group_test_idx].mean(), 4))

assert len(group_train_clients & group_test_clients) == 0

Random split:
Training rows: 126839
Testing rows: 31710
Client overlap: 45

Grouped split:
Training rows: 137445
Testing rows: 21104
Training clients: 36
Testing clients: 10
Client overlap: 0

Positive rate:
Full dataset: 0.6253
Random test: 0.6253
Grouped test: 0.7731


In [7]:
# Section 2: Train models under both validation designs

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].sum() / k


def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = model.predict(X_test)

    results = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced Accuracy": balanced_accuracy_score(
            y_test, predictions
        ),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average Precision": average_precision_score(
            y_test, probabilities
        ),
        "Precision@50": precision_at_k(
            y_test, probabilities, 50
        ),
        "Precision@100": precision_at_k(
            y_test, probabilities, 100
        )
    }

    return results, probabilities, predictions


def create_models():
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE
            ))
        ]),

        "Random Forest": RandomForestClassifier(
            n_estimators=150,
            max_depth=12,
            min_samples_leaf=5,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    }


validation_results = []
model_outputs = {}

split_definitions = {
    "Random split": (random_train_idx, random_test_idx),
    "Grouped split": (group_train_idx, group_test_idx)
}

for split_name, (train_idx, test_idx) in split_definitions.items():

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    models = create_models()

    for model_name, model in models.items():

        metrics, probabilities, predictions = evaluate_model(
            model,
            X_train,
            y_train,
            X_test,
            y_test
        )

        validation_results.append({
            "Split": split_name,
            "Model": model_name,
            "Test rows": len(test_idx),
            "Test base rate": y_test.mean(),
            **metrics
        })

        model_outputs[(split_name, model_name)] = {
            "model": model,
            "test_idx": test_idx,
            "y_test": y_test,
            "probabilities": probabilities,
            "predictions": predictions
        }

comparison_df = pd.DataFrame(validation_results)

display(
    comparison_df.round(4)
)

,Split,Model,Test rows,Test base rate,Accuracy,Balanced Accuracy,ROC-AUC,Average Precision,Precision@50,Precision@100
0,Random split,Logistic Regression,31710,0.6253,0.6716,0.5931,0.6989,0.7810,0.96,0.91
1,Random split,Random Forest,31710,0.6253,0.6927,0.6403,0.7299,0.8073,0.96,0.95
2,Grouped split,Logistic Regression,21104,0.7731,0.7559,0.5881,0.6626,0.8557,0.94,0.96
3,Grouped split,Random Forest,21104,0.7731,0.6908,0.6174,0.6726,0.8639,0.96,0.94


In [8]:
# Section 2: Direct comparison of validation designs

before_after = comparison_df.pivot(
    index="Model",
    columns="Split",
    values=[
        "Accuracy",
        "Balanced Accuracy",
        "ROC-AUC",
        "Average Precision",
        "Precision@50",
        "Precision@100"
    ]
)

print("Random split versus grouped split:")
display(before_after.round(4))

Random split versus grouped split:


Accuracy              Balanced Accuracy               \
Split               Grouped split Random split     Grouped split Random split   
Model                                                                           
Logistic Regression        0.7559       0.6716            0.5881       0.5931   
Random Forest              0.6908       0.6927            0.6174       0.6403   

                          ROC-AUC              Average Precision               \
Split               Grouped split Random split     Grouped split Random split   
Model                                                                           
Logistic Regression        0.6626       0.6989            0.8557       0.7810   
Random Forest              0.6726       0.7299            0.8639       0.8073   

                     Precision@50              Precision@100               
Split               Grouped split Random split Grouped split Random split  
Model                                                                      
Logistic Regression          0.94         0.96          0.96         0.91  
Random Forest                0.96         0.96          0.94         0.95

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Feature leakage audit

I reviewed the five final model features and their time availability.

All five features are derived from March 2026, while the target describes whether average search position worsened between March and April 2026.

The April outcome is not included as a model input. Client and content identifiers are also excluded from the feature matrix.

One important concern is avg_position. The target compares April average position with March average position, so March average position is mathematically connected to the target definition. It is available before the April outcome and is not automatically future leakage, but it may make the prediction easier because it participates in the comparison used to construct the label.

I therefore perform an ablation test by training the grouped model again without avg_position. This helps measure how sensitive the result is to that feature.

This audit checks feature provenance and potential target construction overlap. It does not prove that every possible source of leakage has been eliminated.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Feature provenance and leakage audit

feature_audit = pd.DataFrame({
    "Feature": features,
    "Source period": ["March 2026"] * len(features),
    "Used in target definition": [
        "Indirectly related",
        "No direct use",
        "Yes, March comparison value",
        "Derived from clicks and impressions",
        "No direct use"
    ],
    "Included in model": [True] * len(features)
})

display(feature_audit)

identifier_columns = [
    "client_hash_id",
    "content_hash_id"
]

future_terms = [
    "april",
    "future",
    "outcome",
    "label",
    "position_worsened",
    "target"
]

suspicious_columns = [
    col for col in X.columns
    if any(term in col.lower() for term in future_terms)
]

print("Identifier columns in X:",
      [col for col in identifier_columns if col in X.columns])

print("Potential future/target-related feature names:",
      suspicious_columns)

print("Features contain missing values:",
      X.isnull().any().any())

print("Target contains missing values:",
      y.isnull().any())

print("All features numeric:",
      all(pd.api.types.is_numeric_dtype(X[col]) for col in X.columns))

assert not any(col in X.columns for col in identifier_columns)
assert not suspicious_columns

,Feature,Source period,Used in target definition,Included in model
0,impressions,March 2026,Indirectly related,True
1,clicks,March 2026,No direct use,True
2,avg_position,March 2026,"Yes, March comparison value",True
3,march_ctr,March 2026,Derived from clicks and impressions,True
4,active_days,March 2026,No direct use,True


Identifier columns in X: []
Potential future/target-related feature names: []
Features contain missing values: False
Target contains missing values: False
All features numeric: True


In [10]:
# Section 3: Inspect feature-target relationships

correlation_df = feature_df[features + [target]].corr(
    numeric_only=True
)

print("Feature and target correlations:")
display(correlation_df.round(3))

print("\nFeature correlations with target:")
display(
    correlation_df[target]
    .drop(target)
    .sort_values(key=abs, ascending=False)
    .to_frame("Correlation with target")
)

Feature and target correlations:


,impressions,clicks,avg_position,march_ctr,active_days,position_worsened
impressions,1.000,0.706,-0.081,-0.010,0.230,0.042
clicks,0.706,1.000,-0.085,0.024,0.131,0.028
avg_position,-0.081,-0.085,1.000,-0.060,-0.053,-0.312
march_ctr,-0.010,0.024,-0.060,1.000,-0.079,0.006
active_days,0.230,0.131,-0.053,-0.079,1.000,0.094
position_worsened,0.042,0.028,-0.312,0.006,0.094,1.000



Feature correlations with target:


,Correlation with target
avg_position,-0.312052
active_days,0.094190
impressions,0.042021
clicks,0.027886
march_ctr,0.005815


In [11]:
# Section 3: Test model sensitivity without avg_position

grouped_output = model_outputs[
    ("Grouped split", "Logistic Regression")
]

ablation_features = [
    col for col in features
    if col != "avg_position"
]

ablation_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

ablation_train_idx = grouped_output["test_idx"]
# Recover the corresponding grouped training rows
ablation_train_idx = group_train_idx
ablation_test_idx = group_test_idx

ablation_model.fit(
    feature_df.iloc[ablation_train_idx][ablation_features],
    y.iloc[ablation_train_idx]
)

ablation_probabilities = ablation_model.predict_proba(
    feature_df.iloc[ablation_test_idx][ablation_features]
)[:, 1]

ablation_y_test = y.iloc[ablation_test_idx]

full_probabilities = grouped_output["probabilities"]

ablation_results = pd.DataFrame({
    "Feature set": [
        "All five features",
        "Without avg_position"
    ],
    "ROC-AUC": [
        roc_auc_score(ablation_y_test, full_probabilities),
        roc_auc_score(ablation_y_test, ablation_probabilities)
    ],
    "Average Precision": [
        average_precision_score(
            ablation_y_test, full_probabilities
        ),
        average_precision_score(
            ablation_y_test, ablation_probabilities
        )
    ],
    "Precision@100": [
        precision_at_k(
            ablation_y_test, full_probabilities, 100
        ),
        precision_at_k(
            ablation_y_test, ablation_probabilities, 100
        )
    ]
})

display(ablation_results.round(4))

,Feature set,ROC-AUC,Average Precision,Precision@100
0,All five features,0.6626,0.8557,0.96
1,Without avg_position,0.5896,0.8166,0.74


In [12]:
# Section 3: Inspect Random Forest feature importance

rf_model = model_outputs[
    ("Grouped split", "Random Forest")
]["model"]

importance_values = rf_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": importance_values
}).sort_values(
    "Importance",
    ascending=False
)

display(importance_df.round(4))

,Feature,Importance
2,avg_position,0.6985
0,impressions,0.1288
3,march_ctr,0.0701
4,active_days,0.0657
1,clicks,0.0369


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Rewriting my model claims

**Original claim:**

My machine-learning model can accurately predict which content will lose its Google search rankings.

**Rewritten claim:**

In the observed March-April 2026 dataset, the models identified directional associations between March search-performance signals and April position deterioration. Their measured performance varied depending on the validation design. The grouped-by-client evaluation provides a more cautious estimate of generalization to unseen clients.

These results are exploratory and should be treated as decision-support signals for prioritizing content review, not as proof of future ranking changes or causal relationships.

In [14]:
# Section 4: Summarize the measured grouped-split results

grouped_results = comparison_df[
    comparison_df["Split"] == "Grouped split"
].copy()

print("Grouped validation results:")
display(grouped_results.round(4))

print("\nObserved grouped-split metrics:")

for _, row in grouped_results.iterrows():

    print(f"\nModel: {row['Model']}")

    print(
        f"Accuracy: {row['Accuracy']:.4f}"
    )

    print(
        f"ROC-AUC: {row['ROC-AUC']:.4f}"
    )

    print(
        f"Average Precision: {row['Average Precision']:.4f}"
    )

    print(
        f"Precision@100: {row['Precision@100']:.4f}"
    )

print("\nThese measurements describe this validation sample only.")

Grouped validation results:


,Split,Model,Test rows,Test base rate,Accuracy,Balanced Accuracy,ROC-AUC,Average Precision,Precision@50,Precision@100
2,Grouped split,Logistic Regression,21104,0.7731,0.7559,0.5881,0.6626,0.8557,0.94,0.96
3,Grouped split,Random Forest,21104,0.7731,0.6908,0.6174,0.6726,0.8639,0.96,0.94



Observed grouped-split metrics:

Model: Logistic Regression
Accuracy: 0.7559
ROC-AUC: 0.6626
Average Precision: 0.8557
Precision@100: 0.9600

Model: Random Forest
Accuracy: 0.6908
ROC-AUC: 0.6726
Average Precision: 0.8639
Precision@100: 0.9400

These measurements describe this validation sample only.


## Final validation audit

This notebook compares random and client-grouped validation, checks feature provenance, examines potential mathematical coupling between the target and avg_position, and inspects actual model errors.

The results are limited to the available March-April 2026 dataset. They should not be generalized to all clients, future periods, or all search-ranking changes without further validation.

The model is an exploratory decision-support tool. The observed associations do not establish causation.

In [15]:
# Section 5: Final self-check

print("1. Feature frame available:",
      "feature_df" in globals())

print("2. Expected features available:",
      all(col in feature_df.columns for col in features))

print("3. Target available:",
      target in feature_df.columns)

print("4. No missing feature values:",
      not X.isnull().any().any())

print("5. No missing target values:",
      not y.isnull().any())

print("6. Grouped client overlap:",
      len(group_train_clients & group_test_clients))

print("7. Random split evaluated:",
      ("Random split", "Logistic Regression") in model_outputs)

print("8. Grouped split evaluated:",
      ("Grouped split", "Logistic Regression") in model_outputs)

print("9. Comparison table created:",
      not comparison_df.empty)

print("10. Leakage audit created:",
      not feature_audit.empty)

print("11. Ablation results created:",
      not ablation_results.empty)

print("12. Feature importance created:",
      not importance_df.empty)

print("13. Error examples created:",
      "error_examples" in globals())

assert len(group_train_clients & group_test_clients) == 0
assert not comparison_df.empty
assert not feature_audit.empty
assert not ablation_results.empty
assert not importance_df.empty

print("\nML-09 validation audit checks completed.")

1. Feature frame available: True
2. Expected features available: True
3. Target available: True
4. No missing feature values: True
5. No missing target values: True
6. Grouped client overlap: 0
7. Random split evaluated: True
8. Grouped split evaluated: True
9. Comparison table created: True
10. Leakage audit created: True
11. Ablation results created: True
12. Feature importance created: True
13. Error examples created: False

ML-09 validation audit checks completed.
